# **Tarea MLFlow & Databricks**

---

In [1]:
import os, mlflow
import pickle
import pandas as pd
from sklearn.metrics import  root_mean_squared_error
from sklearn.feature_extraction import  DictVectorizer
import os, mlflow
from dotenv import load_dotenv
import math
import optuna
import pathlib
from optuna.samplers import TPESampler
from mlflow.models.signature import infer_signature
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from mlflow import MlflowClient
from datetime import datetime
import mlflow.pyfunc

c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\nyc-taxi-predictions-2025\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)  # Carga las variables del archivo .env
EXPERIMENT_NAME = "/Users/monica.ibarra@iteso.mx/nyc-taxi-experiments"

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

## **Train model**

---

In [4]:
def read_dataframe(filename):

    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [6]:
df_train = read_dataframe('../data/green_tripdata_2025-01.parquet')
df_val = read_dataframe('../data/green_tripdata_2025-02.parquet')

In [7]:
def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)

df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
categorical = ['PU_DO']
numerical = ['trip_distance']
dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

X_val = preprocess(df_val, dv)

In [ ]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [ ]:
training_dataset = mlflow.data.from_numpy(X_train.data, targets=y_train, name="green_tripdata_2025-01")
validation_dataset = mlflow.data.from_numpy(X_val.data, targets=y_val, name="green_tripdata_2025-02")

## **Gradient Boost**

---

#### **Función objetivo**

In [ ]:
# ------------------------------------------------------------
# Definir la función objetivo para Optuna
#    - Recibe un `trial`, que se usa para proponer hiperparámetros.
#    - Entrena un modelo con esos hiperparámetros.
#    - Calcula la métrica de validación (RMSE) y la retorna (Optuna la minimizará).
#    - Abrimos un run anidado de MLflow para registrar cada trial.
# ------------------------------------------------------------
def objective_gb(trial: optuna.trial.Trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", math.exp(-7), 0.3, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 10, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "random_state": 42,
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "gradient_boosting")
        mlflow.log_params(params)

        model = GradientBoostingRegressor(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

        signature = infer_signature(X_val, y_pred)
        mlflow.sklearn.log_model(model, "model", input_example=X_val[:5], signature=signature)

    return rmse

#### **Flujo de búsqueda**

In [ ]:
mlflow.sklearn.autolog(log_models=False)

# ------------------------------------------------------------
# Crear el estudio de Optuna
#    - Usamos TPE (Tree-structured Parzen Estimator) como sampler.
#    - direction="minimize" porque queremos minimizar el RMSE.
# ------------------------------------------------------------
sampler = TPESampler(seed=42)
study_gb = optuna.create_study(direction="minimize", sampler=sampler)

# ------------------------------------------------------------
# Ejecutar la optimización (n_trials = número de intentos)
#    - Cada trial ejecuta la función objetivo con un set distinto de hiperparámetros.
#    - Abrimos un run "padre" para agrupar toda la búsqueda.
# ------------------------------------------------------------
with mlflow.start_run(run_name="GradientBoost Hyperparameter Optimization (Optuna)", nested=True):
    study_gb.optimize(objective_gb, n_trials=10)

    # --------------------------------------------------------
    # Recuperar y registrar los mejores hiperparámetros
    # --------------------------------------------------------
    best_params = study_gb.best_params

    mlflow.log_params(best_params)

    # Etiquetas del run "padre" (metadatos del experimento)
    mlflow.set_tags({
        "project": "NYC Taxi Time Prediction Project",
        "optimizer_engine": "optuna",
        "model_family": "gradient_boosting",
        "feature_set_version": 1,
    })

    final_model = GradientBoostingRegressor(**best_params)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    feature_names = dv.get_feature_names_out()
    input_example = pd.DataFrame(X_val[:5].toarray(), columns=feature_names)
    signature = infer_signature(input_example, y_val[:5])

    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)

[I 2025-10-27 21:19:55,363] A new study created in memory with name: no-name-4830b0f2-b416-49b2-97ba-e630b15f2dd5
2025/10/27 21:20:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:20:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:20:46 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:20:50,825] Trial 0 finished with value: 6.496399816815654 and parameters: {'learning_rate': 0.007993270448118463, 'max_leaf_nodes': 77, 'max_depth': 10, 'min_samples_leaf': 12}. Best is trial 0 with value: 6.496399816815654.


🏃 View run angry-roo-424 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/f20532abc99e4751a2378ff4a39ec8bc
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:21:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:21:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:21:11 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:21:14,603] Trial 1 finished with value: 8.125801155365895 and parameters: {'learning_rate': 0.0022525064230539864, 'max_leaf_nodes': 21, 'max_depth': 3, 'min_samples_leaf': 18}. Best is trial 0 with value: 6.496399816815654.


🏃 View run clean-moose-177 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/f56ab97b9edf466ea79569524f598065
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:21:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:21:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:21:34 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:21:37,547] Trial 2 finished with value: 5.716773865305269 and parameters: {'learning_rate': 0.029720416526464566, 'max_leaf_nodes': 60, 'max_depth': 3, 'min_samples_leaf': 20}. Best is trial 2 with value: 5.716773865305269.


...

🏃 View run charming-lynx-985 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/2afb5799fbec42e59c54b9a654f60a1b
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:24:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:25:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:25:02 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:25:05,677] Trial 9 finished with value: 7.11299850468749 and parameters: {'learning_rate': 0.00532975339252159, 'max_leaf_nodes': 16, 'max_depth': 9, 'min_samples_leaf': 9}. Best is trial 3 with value: 5.538795759490097.


🏃 View run trusting-rat-246 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/41332714217d4b32a3a464c204bb5f93
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:25:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:25:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\nyc-taxi-predictions-2025\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(
2025/10/27 21:25:35 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run GradientBoost Hyperparameter Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/ed9cc72d77d94d7e9303a5d83d274208
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


## **Random Forest**

---

#### **Función objetivo**

In [ ]:
# ------------------------------------------------------------
# Definir la función objetivo para Optuna
#    - Recibe un `trial`, que se usa para proponer hiperparámetros.
#    - Entrena un modelo con esos hiperparámetros.
#    - Calcula la métrica de validación (RMSE) y la retorna (Optuna la minimizará).
#    - Abrimos un run anidado de MLflow para registrar cada trial.
# ------------------------------------------------------------
def objective_rf(trial: optuna.trial.Trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 30, 80),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 5, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

    # Run anidado para dejar rastro de cada trial en MLflow
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "random_forest")
        mlflow.log_params(params)

        model = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

        signature = infer_signature(X_val, y_pred)
        mlflow.sklearn.log_model(model, "model", input_example=X_val[:5], signature=signature)

    return rmse

#### **Flujo de búsqueda**

In [ ]:
mlflow.sklearn.autolog(log_models=False)

# ------------------------------------------------------------
# Ejecutar la optimización (n_trials = número de intentos)
#    - Cada trial ejecuta la función objetivo con un set distinto de hiperparámetros.
#    - Abrimos un run "padre" para agrupar toda la búsqueda.
# ------------------------------------------------------------
study_rf = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
with mlflow.start_run(run_name="RandomForest Hyperparameter Optimization (Optuna)", nested=True):
    study_rf.optimize(objective_rf, n_trials=10)

    # --------------------------------------------------------
    # Recuperar y registrar los mejores hiperparámetros
    # --------------------------------------------------------
    best_params_rf = study_rf.best_params

    mlflow.log_params(best_params_rf)

    # Etiquetas del run "padre" (metadatos del experimento)
    mlflow.set_tags({
        "project": "NYC Taxi Time Prediction Project",
        "optimizer_engine": "optuna",
        "model_family": "random_forest",
        "feature_set_version": 1,
    })

    mlflow.sklearn.autolog(log_models=False)

    # Entrenar modelo final con mejores hiperparámetros
    final_model = RandomForestRegressor(**best_params_rf, n_jobs=-1, random_state=42)
    final_model.fit(X_train, y_train)
    y_pred = final_model.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    feature_names = dv.get_feature_names_out()
    input_example = pd.DataFrame(X_val[:5].toarray(), columns=feature_names)
    signature = infer_signature(input_example, y_val[:5])

    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)


[I 2025-10-27 21:40:12,117] A new study created in memory with name: no-name-ad7db413-9d36-4f0d-9ff8-e4fa0ee6b2cb
2025/10/27 21:40:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:40:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:40:30 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:40:35,283] Trial 0 finished with value: 7.192333875814888 and parameters: {'n_estimators': 49, 'max_depth': 39, 'min_samples_split': 16, 'max_features': 'sqrt'}. Best is trial 0 with value: 7.192333875814888.


🏃 View run skittish-newt-29 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/661f84b2009c4781abe54d08a7b58509
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:40:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:40:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:40:55 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:41:06,516] Trial 1 finished with value: 5.481438540048821 and parameters: {'n_estimators': 32, 'max_depth': 36, 'min_samples_split': 14, 'max_features': None}. Best is trial 1 with value: 5.481438540048821.


🏃 View run carefree-mink-600 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/e8cef67e40f049b69a6e3fe3e9831f04
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:41:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:41:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:41:25 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:41:30,816] Trial 2 finished with value: 5.580631009827177 and parameters: {'n_estimators': 72, 'max_depth': 12, 'min_samples_split': 7, 'max_features': None}. Best is trial 1 with value: 5.481438540048821.


...

🏃 View run upset-dog-52 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/d16b40043805419aadc5deab93755adf
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:44:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:44:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/10/27 21:44:46 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-10-27 21:45:08,513] Trial 9 finished with value: 5.489059947767261 and parameters: {'n_estimators': 60, 'max_depth': 38, 'min_samples_split': 6, 'max_features': None}. Best is trial 1 with value: 5.481438540048821.


🏃 View run silent-ray-609 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/d882ef60922044a9b04476b5a99c9baa
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


2025/10/27 21:45:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/10/27 21:45:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\nyc-taxi-predictions-2025\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
2025/10/27 21:45:38 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run RandomForest Hyperparameter Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855/runs/65ffa9a80eba4e13832a7be734e96c7e
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/3821134567092855


## **Registrar el modelo**

---

In [ ]:
model_name = "workspace.default.nyc-taxi-model"

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.rmse ASC"],
    output_format="list"
)

# Obtener el mejor run
if len(runs) > 0:
    best_run = runs[0]
    print("🏆 Champion Run encontrado:")
    print(f"Run ID: {best_run.info.run_id}")
    print(f"Validation RMSE: {best_run.data.metrics.get('rmse')}")
    print(f"Params: {best_run.data.params}")
else:
    print("⚠️ No se encontraron runs con métrica RMSE.")

🏆 Champion Run encontrado:
Run ID: e8cef67e40f049b69a6e3fe3e9831f04
Validation RMSE: 5.481438540048821
Params: {'bootstrap': 'True', 'ccp_alpha': '0.0', 'criterion': 'squared_error', 'max_depth': '36', 'max_features': 'None', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '1', 'min_samples_split': '14', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '32', 'n_jobs': '-1', 'oob_score': 'False', 'random_state': '42', 'verbose': '0', 'warm_start': 'False'}


In [ ]:
run_id = best_run.info.run_id

In [ ]:
result = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_name
)

Registered model 'workspace.default.nyc-taxi-model' already exists. Creating a new version of this model...
2025/10/27 21:46:03 WARNING mlflow.tracking._model_registry.fluent: Run with id e8cef67e40f049b69a6e3fe3e9831f04 has no artifacts at artifact path 'model', registering model based on models:/m-3ae42873395b42f6a2fd64cdc7d6dca9 instead
Uploading artifacts: 100%|██████████| 9/9 [00:04<00:00,  2.10it/s]
Created version '10' of model 'workspace.default.nyc-taxi-model'.


## **Asignar alias challenger**

---

In [ ]:
client = MlflowClient()

model_version = result.version
new_alias = "Challenger"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)

date = datetime.today()

client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1761623171511, current_stage=None, deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description=('The model version 10 was transitioned to Challenger on 2025-10-27 '
 '21:46:16.350185'), last_updated_timestamp=1761623179073, metrics=[<Metric: dataset_digest='', dataset_name='', key='rmse', model_id='m-3ae42873395b42f6a2fd64cdc7d6dca9', run_id='e8cef67e40f049b69a6e3fe3e9831f04', step=0, timestamp=1761622847265, value=5.481438540048821>,
 <Metric: dataset_digest='', dataset_name='', key='training_mean_absolute_error', model_id='m-3ae42873395b42f6a2fd64cdc7d6dca9', run_id='e8cef67e40f049b69a6e3fe3e9831f04', step=0, timestamp=1761622843606, value=2.8996472364832995>,
 <Metric: dataset_digest='', dataset_name='', key='training_mean_squared_error', model_id='m-3ae42873395b42f6a2fd64cdc7d6dca9

## **Datos de marzo de 2025**

In [8]:
df_val = read_dataframe('../data/green_tripdata_2025-03.parquet')

In [9]:
from mlflow import MlflowClient
client = MlflowClient()

model_name = "workspace.default.nyc-taxi-model"   # ajusta si hace falta
# obtener la versión asociada al alias 'Champion' o 'Challenger'
champ = client.get_model_version_by_alias(model_name, "Champion")
chall = client.get_model_version_by_alias(model_name, "Challenger")
print("Champion:", champ.version, "run_id:", champ.run_id)
print("Challenger:", chall.version, "run_id:", chall.run_id)


Champion: 14 run_id: fa660b093a254833b2e23cbb8e5c5b9d
Challenger: 10 run_id: e8cef67e40f049b69a6e3fe3e9831f04


In [ ]:
with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

# Preprocesar el dataset de marzo
X_march = preprocess(df_val, dv)
y_march = df_val['duration'].values
X_march_arr = X_march.toarray()

# Cargar Champion
champion_uri = f"models:/{model_name}@Champion"
champion_model = mlflow.pyfunc.load_model(champion_uri)

# Predecir
champion_preds = champion_model.predict(X_march_arr)
champion_rmse = root_mean_squared_error(y_march, champion_preds)
print(f"Champion RMSE (marzo): {champion_rmse:.6f}")

c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\nyc-taxi-predictions-2025\.venv\Lib\site-packages\mlflow\xgboost\__init__.py:321: UserWarning: [10:52:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  model.load_model(xgb_model_path)


Champion RMSE (marzo): 23.951918


In [12]:
# Cargar el preprocesador que sí tienes
with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

# Preprocesar el dataset de marzo
X_march = preprocess(df_val, dv)
y_march = df_val['duration'].values
X_march_arr = X_march.toarray()

# Cargar Challenger
challenger_uri = f"models:/{model_name}@Challenger"
challenger_model = mlflow.pyfunc.load_model(challenger_uri)

# Predecir
challenger_preds = challenger_model.predict(X_march_arr)
challenger_rmse = root_mean_squared_error(y_march, challenger_preds)
print(f"Challenger RMSE (marzo): {challenger_rmse:.6f}")


Challenger RMSE (marzo): 6.076449
